In [1]:
fact_interaction_df = spark.table(
    "demo.gold.fact_learning_interaction"
)

fact_attempt_df = spark.table(
    "demo.gold.fact_practice_attempt"
)

fact_session_df = spark.table(
    "demo.gold.fact_learning_session"
)

fact_feedback_df = spark.table(
    "demo.gold.fact_learning_feedback"
)

fact_validation_df = spark.table(
    "demo.gold.fact_ai_insight_validation"
)

fact_concept_state_df = spark.table(
    "demo.gold.fact_learner_concept_state"
)

print("Interactions:", fact_interaction_df.count())
print("Attempts:", fact_attempt_df.count())
print("Sessions:", fact_session_df.count())
print("Feedback:", fact_feedback_df.count())
print("AI validations:", fact_validation_df.count())
print("Concept states:", fact_concept_state_df.count())

Interactions: 6
Attempts: 5
Sessions: 3
Feedback: 11
AI validations: 9
Concept states: 12


In [2]:
from pyspark.sql.functions import (
    col,
    to_date
)

interaction_dates_df = (
    fact_interaction_df
    .select(
        "user_key",
        col("event_date").alias("date")
    )
)

session_dates_df = (
    fact_session_df
    .select(
        "user_key",
        to_date("session_start_time").alias("date")
    )
)

feedback_dates_df = (
    fact_feedback_df
    .select(
        "user_key",
        to_date("feedback_time").alias("date")
    )
)

validation_dates_df = (
    fact_validation_df
    .select(
        "user_key",
        to_date("validation_time").alias("date")
    )
)

concept_state_dates_df = (
    fact_concept_state_df
    .select(
        "user_key",
        to_date("valid_from").alias("date")
    )
)

learner_daily_spine_df = (
    interaction_dates_df
    .unionByName(session_dates_df)
    .unionByName(feedback_dates_df)
    .unionByName(validation_dates_df)
    .unionByName(concept_state_dates_df)
    .filter(col("date").isNotNull())
    .distinct()
)

learner_daily_spine_df.orderBy(
    "user_key",
    "date"
).show(truncate=False)

print(
    "Learner-date rows:",
    learner_daily_spine_df.count()
)

+--------+----------+
|user_key|date      |
+--------+----------+
|1       |2026-07-20|
|1       |2026-07-23|
|1       |2026-07-28|
|2       |2026-07-21|
|2       |2026-07-24|
|2       |2026-07-28|
|3       |2026-07-22|
|3       |2026-07-25|
|3       |2026-07-28|
+--------+----------+

Learner-date rows: 9


In [3]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    to_date
)

daily_sessions_df = (
    fact_session_df
    .groupBy(
        "user_key",
        to_date("session_start_time").alias("date")
    )
    .agg(
        countDistinct("session_id")
        .cast("int")
        .alias("total_sessions")
    )
)

daily_interactions_df = (
    fact_interaction_df
    .groupBy(
        "user_key",
        col("event_date").alias("date")
    )
    .agg(
        countDistinct("event_id")
        .cast("int")
        .alias("total_learning_events")
    )
)

daily_practice_topics_df = (
    fact_attempt_df
    .groupBy(
        "user_key",
        to_date("attempt_time").alias("date")
    )
    .agg(
        countDistinct("topic_key")
        .cast("int")
        .alias("topics_practiced")
    )
)

In [4]:
print("Daily sessions")
daily_sessions_df.orderBy(
    "user_key",
    "date"
).show(truncate=False)

print("Daily learning events")
daily_interactions_df.orderBy(
    "user_key",
    "date"
).show(truncate=False)

print("Daily practiced topics")
daily_practice_topics_df.orderBy(
    "user_key",
    "date"
).show(truncate=False)

Daily sessions
+--------+----------+--------------+
|user_key|date      |total_sessions|
+--------+----------+--------------+
|1       |2026-07-20|1             |
|2       |2026-07-21|1             |
|3       |2026-07-22|1             |
+--------+----------+--------------+

Daily learning events
+--------+----------+---------------------+
|user_key|date      |total_learning_events|
+--------+----------+---------------------+
|1       |2026-07-20|2                    |
|2       |2026-07-21|2                    |
|3       |2026-07-22|2                    |
+--------+----------+---------------------+

Daily practiced topics
+--------+----------+----------------+
|user_key|date      |topics_practiced|
+--------+----------+----------------+
|1       |2026-07-20|1               |
|2       |2026-07-21|1               |
|3       |2026-07-22|1               |
+--------+----------+----------------+



In [5]:
from pyspark.sql.functions import (
    col,
    to_date,
    avg,
    sum,
    when
)

daily_concept_state_df = (
    fact_concept_state_df
    .filter(col("is_current") == True)
    .groupBy(
        "user_key",
        to_date("valid_from").alias("date")
    )
    .agg(
        sum(
            when(
                col("mastery_score").isNotNull()
                & (col("mastery_score") < 0.6),
                1
            ).otherwise(0)
        )
        .cast("int")
        .alias("weak_topics_count"),

        sum("repeated_mistake_count")
        .cast("int")
        .alias("repeated_mistakes_count"),

        avg("mastery_score")
        .cast("float")
        .alias("avg_mastery_score"),

        sum(
            when(
                col("struggle_risk_score").isNotNull()
                & (col("struggle_risk_score") >= 0.6),
                1
            ).otherwise(0)
        )
        .cast("int")
        .alias("at_risk_topics_count")
    )
)

daily_concept_state_df.orderBy(
    "user_key",
    "date"
).show(truncate=False)

+--------+----------+-----------------+-----------------------+-----------------+--------------------+
|user_key|date      |weak_topics_count|repeated_mistakes_count|avg_mastery_score|at_risk_topics_count|
+--------+----------+-----------------+-----------------------+-----------------+--------------------+
|1       |2026-07-23|0                |0                      |0.7              |0                   |
|1       |2026-07-28|1                |0                      |0.4567           |1                   |
|2       |2026-07-28|1                |0                      |0.5667           |0                   |
|3       |2026-07-22|0                |0                      |0.9              |0                   |
|3       |2026-07-25|0                |0                      |0.6              |0                   |
|3       |2026-07-28|0                |0                      |0.9              |0                   |
+--------+----------+-----------------+-----------------------+----------

In [6]:
from pyspark.sql.functions import (
    col,
    to_date,
    avg
)

daily_validation_df = (
    fact_validation_df
    .groupBy(
        "user_key",
        to_date("validation_time").alias("date")
    )
    .agg(
        avg("reliability_score")
        .cast("float")
        .alias("avg_reliability_score")
    )
)

daily_validation_df.orderBy(
    "user_key",
    "date"
).show(truncate=False)

+--------+----------+---------------------+
|user_key|date      |avg_reliability_score|
+--------+----------+---------------------+
|1       |2026-07-28|0.69666666           |
|2       |2026-07-28|0.63666666           |
|3       |2026-07-28|0.65                 |
+--------+----------+---------------------+



In [7]:
from pyspark.sql.functions import (
    col,
    to_date,
    avg
)

daily_feedback_df = (
    fact_feedback_df
    .filter(
        col("feedback_stage") == "general_check_in"
    )
    .groupBy(
        "user_key",
        to_date("feedback_time").alias("date")
    )
    .agg(
        avg("motivation_score")
        .cast("float")
        .alias("avg_motivation_score"),

        avg("stress_score")
        .cast("float")
        .alias("avg_stress_score"),

        avg("perceived_understanding_score")
        .cast("float")
        .alias("avg_self_reported_understanding")
    )
)

daily_feedback_df.orderBy(
    "user_key",
    "date"
).show(truncate=False)

+--------+----------+--------------------+----------------+-------------------------------+
|user_key|date      |avg_motivation_score|avg_stress_score|avg_self_reported_understanding|
+--------+----------+--------------------+----------------+-------------------------------+
|1       |2026-07-23|6.0                 |7.0             |5.5                            |
|2       |2026-07-24|5.0                 |6.0             |5.0                            |
|3       |2026-07-25|8.0                 |3.0             |7.5                            |
+--------+----------+--------------------+----------------+-------------------------------+



In [8]:
from pyspark.sql.functions import col, coalesce, lit

agg_learner_overview_daily_df = (
    learner_daily_spine_df.alias("spine")

    .join(
        daily_sessions_df.alias("sessions"),
        ["user_key", "date"],
        "left"
    )

    .join(
        daily_interactions_df.alias("interactions"),
        ["user_key", "date"],
        "left"
    )

    .join(
        daily_practice_topics_df.alias("practice"),
        ["user_key", "date"],
        "left"
    )

    .join(
        daily_concept_state_df.alias("state"),
        ["user_key", "date"],
        "left"
    )

    .join(
        daily_validation_df.alias("validation"),
        ["user_key", "date"],
        "left"
    )

    .join(
        daily_feedback_df.alias("feedback"),
        ["user_key", "date"],
        "left"
    )

    .select(
        col("user_key").cast("int"),
        col("date"),

        coalesce(
            col("total_sessions"),
            lit(0)
        ).cast("int").alias("total_sessions"),

        coalesce(
            col("total_learning_events"),
            lit(0)
        ).cast("int").alias("total_learning_events"),

        coalesce(
            col("topics_practiced"),
            lit(0)
        ).cast("int").alias("topics_practiced"),

        coalesce(
            col("weak_topics_count"),
            lit(0)
        ).cast("int").alias("weak_topics_count"),

        coalesce(
            col("repeated_mistakes_count"),
            lit(0)
        ).cast("int").alias("repeated_mistakes_count"),

        col("avg_mastery_score").cast("float"),

        coalesce(
            col("at_risk_topics_count"),
            lit(0)
        ).cast("int").alias("at_risk_topics_count"),

        col("avg_reliability_score").cast("float"),
        col("avg_motivation_score").cast("float"),
        col("avg_stress_score").cast("float"),
        col("avg_self_reported_understanding").cast("float")
    )
)

agg_learner_overview_daily_df.orderBy(
    "user_key",
    "date"
).show(truncate=False)

+--------+----------+--------------+---------------------+----------------+-----------------+-----------------------+-----------------+--------------------+---------------------+--------------------+----------------+-------------------------------+
|user_key|date      |total_sessions|total_learning_events|topics_practiced|weak_topics_count|repeated_mistakes_count|avg_mastery_score|at_risk_topics_count|avg_reliability_score|avg_motivation_score|avg_stress_score|avg_self_reported_understanding|
+--------+----------+--------------+---------------------+----------------+-----------------+-----------------------+-----------------+--------------------+---------------------+--------------------+----------------+-------------------------------+
|1       |2026-07-20|1             |2                    |1               |0                |0                      |NULL             |0                   |NULL                 |NULL                |NULL            |NULL                           |
|1  

In [9]:
spark.sql("""
DELETE FROM demo.gold.agg_learner_overview_daily
""")

DataFrame[]

In [10]:
agg_learner_overview_daily_df.writeTo(
    "demo.gold.agg_learner_overview_daily"
).append()

In [11]:
spark.sql("""
SELECT *
FROM demo.gold.agg_learner_overview_daily
ORDER BY user_key, date
""").show(truncate=False)

+--------+----------+--------------+---------------------+----------------+-----------------+-----------------------+-----------------+--------------------+---------------------+--------------------+----------------+-------------------------------+
|user_key|date      |total_sessions|total_learning_events|topics_practiced|weak_topics_count|repeated_mistakes_count|avg_mastery_score|at_risk_topics_count|avg_reliability_score|avg_motivation_score|avg_stress_score|avg_self_reported_understanding|
+--------+----------+--------------+---------------------+----------------+-----------------+-----------------------+-----------------+--------------------+---------------------+--------------------+----------------+-------------------------------+
|1       |2026-07-20|1             |2                    |1               |0                |0                      |NULL             |0                   |NULL                 |NULL                |NULL            |NULL                           |
|1  

In [12]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(
        DISTINCT CONCAT(
            CAST(user_key AS STRING),
            '||',
            CAST(date AS STRING)
        )
    ) AS distinct_learner_dates
FROM demo.gold.agg_learner_overview_daily
""").show()

+----------+----------------------+
|total_rows|distinct_learner_dates|
+----------+----------------------+
|         9|                     9|
+----------+----------------------+



In [13]:
from pyspark.sql.functions import (
    col,
    to_date,
    avg,
    sum,
    count,
    when
)

daily_topic_practice_df = (
    fact_attempt_df
    .groupBy(
        "user_key",
        "topic_key",
        to_date("attempt_time").alias("date")
    )
    .agg(
        count("*")
            .cast("int")
            .alias("total_attempts"),

        avg(
            when(
                col("is_correct") == False,
                1.0
            ).otherwise(0.0)
        )
            .cast("float")
            .alias("failure_rate"),

        avg("score")
            .cast("float")
            .alias("avg_score"),

        avg("hints_used")
            .cast("float")
            .alias("avg_hints_used")
    )
)

daily_topic_practice_df.orderBy(
    "user_key",
    "topic_key",
    "date"
).show(truncate=False)

+--------+---------+----------+--------------+------------+---------+--------------+
|user_key|topic_key|date      |total_attempts|failure_rate|avg_score|avg_hints_used|
+--------+---------+----------+--------------+------------+---------+--------------+
|1       |5        |2026-07-20|2             |0.5         |0.5      |0.5           |
|2       |9        |2026-07-21|2             |0.5         |0.5      |0.0           |
|3       |8        |2026-07-22|1             |0.0         |1.0      |0.0           |
+--------+---------+----------+--------------+------------+---------+--------------+



In [14]:
from pyspark.sql.functions import (
    col,
    to_date
)

daily_topic_state_df = (
    fact_concept_state_df
    .filter(
        col("is_current") == True
    )
    .select(
        col("user_key").cast("int"),
        col("topic_key").cast("int"),
        to_date("valid_from").alias("date"),

        col("repeated_mistake_count")
            .cast("int"),

        col("confidence_score")
            .cast("float")
            .alias("state_confidence_score"),

        col("difficulty_score")
            .cast("float")
    )
)

daily_topic_state_df.orderBy(
    "user_key",
    "topic_key",
    "date"
).show(truncate=False)

+--------+---------+----------+----------------------+----------------------+----------------+
|user_key|topic_key|date      |repeated_mistake_count|state_confidence_score|difficulty_score|
+--------+---------+----------+----------------------+----------------------+----------------+
|1       |5        |2026-07-28|0                     |0.4333                |0.675           |
|1       |6        |2026-07-28|0                     |NULL                  |NULL            |
|1       |9        |2026-07-23|0                     |0.7                   |0.0             |
|1       |10       |2026-07-28|0                     |NULL                  |NULL            |
|2       |2        |2026-07-28|0                     |NULL                  |NULL            |
|2       |7        |2026-07-28|0                     |NULL                  |NULL            |
|2       |9        |2026-07-28|0                     |0.6333                |0.6             |
|3       |3        |2026-07-28|0                  

In [15]:
from pyspark.sql.functions import (
    col,
    to_date,
    avg
)

daily_topic_feedback_df = (
    fact_feedback_df
    .filter(
        col("topic_key").isNotNull()
    )
    .groupBy(
        "user_key",
        "topic_key",
        to_date("feedback_time").alias("date")
    )
    .agg(
        avg("confidence_score")
        .cast("float")
        .alias("feedback_confidence_score")
    )
)

daily_topic_feedback_df.orderBy(
    "user_key",
    "topic_key",
    "date"
).show(truncate=False)

+--------+---------+----------+-------------------------+
|user_key|topic_key|date      |feedback_confidence_score|
+--------+---------+----------+-------------------------+
|1       |5        |2026-07-23|4.0                      |
|1       |9        |2026-07-23|7.0                      |
|2       |9        |2026-07-24|5.0                      |
|3       |4        |2026-07-25|9.0                      |
|3       |5        |2026-07-25|6.0                      |
+--------+---------+----------+-------------------------+



In [16]:
topic_practice_dates_df = (
    daily_topic_practice_df
    .select(
        "user_key",
        "topic_key",
        "date"
    )
)

topic_state_dates_df = (
    daily_topic_state_df
    .select(
        "user_key",
        "topic_key",
        "date"
    )
)

topic_feedback_dates_df = (
    daily_topic_feedback_df
    .select(
        "user_key",
        "topic_key",
        "date"
    )
)

learner_topic_daily_spine_df = (
    topic_practice_dates_df
    .unionByName(topic_state_dates_df)
    .unionByName(topic_feedback_dates_df)
    .filter(
        col("user_key").isNotNull()
        & col("topic_key").isNotNull()
        & col("date").isNotNull()
    )
    .distinct()
)

learner_topic_daily_spine_df.orderBy(
    "user_key",
    "topic_key",
    "date"
).show(truncate=False)

print(
    "Learner-topic-date rows:",
    learner_topic_daily_spine_df.count()
)

+--------+---------+----------+
|user_key|topic_key|date      |
+--------+---------+----------+
|1       |5        |2026-07-20|
|1       |5        |2026-07-23|
|1       |5        |2026-07-28|
|1       |6        |2026-07-28|
|1       |9        |2026-07-23|
|1       |10       |2026-07-28|
|2       |2        |2026-07-28|
|2       |7        |2026-07-28|
|2       |9        |2026-07-21|
|2       |9        |2026-07-24|
|2       |9        |2026-07-28|
|3       |3        |2026-07-28|
|3       |4        |2026-07-25|
|3       |4        |2026-07-28|
|3       |5        |2026-07-25|
|3       |8        |2026-07-22|
|3       |10       |2026-07-28|
+--------+---------+----------+

Learner-topic-date rows: 17


In [17]:
from pyspark.sql.functions import (
    col,
    coalesce,
    lit,
    when,
    row_number
)
from pyspark.sql.window import Window

concept_weakness_base_df = (
    learner_topic_daily_spine_df.alias("spine")

    .join(
        daily_topic_practice_df.alias("practice"),
        ["user_key", "topic_key", "date"],
        "left"
    )

    .join(
        daily_topic_state_df.alias("state"),
        ["user_key", "topic_key", "date"],
        "left"
    )

    .join(
        daily_topic_feedback_df.alias("feedback"),
        ["user_key", "topic_key", "date"],
        "left"
    )

    .select(
        col("user_key").cast("int"),
        col("topic_key").cast("int"),
        col("date"),

        col("practice.failure_rate").cast("float"),
        col("practice.avg_score").cast("float"),
        col("practice.avg_hints_used").cast("float"),

        coalesce(
            col("state.repeated_mistake_count"),
            lit(0)
        ).cast("int").alias("repeated_mistake_count"),

        # feedback הוא 1–10; מצב המושג כבר 0–1
        coalesce(
            col("feedback.feedback_confidence_score") / 10.0,
            col("state.state_confidence_score")
        ).cast("float").alias("avg_confidence_score"),

        col("state.difficulty_score").cast("float")
    )
)

In [18]:
concept_weakness_scored_df = (
    concept_weakness_base_df

    .withColumn(
        "weakness_weight_sum",

        when(col("failure_rate").isNotNull(), 0.4).otherwise(0.0)
        +
        when(col("difficulty_score").isNotNull(), 0.3).otherwise(0.0)
        +
        when(col("avg_score").isNotNull(), 0.2).otherwise(0.0)
        +
        when(col("avg_confidence_score").isNotNull(), 0.1).otherwise(0.0)
    )

    .withColumn(
        "weakness_weighted_sum",

        when(
            col("failure_rate").isNotNull(),
            col("failure_rate") * 0.4
        ).otherwise(0.0)

        +

        when(
            col("difficulty_score").isNotNull(),
            col("difficulty_score") * 0.3
        ).otherwise(0.0)

        +

        when(
            col("avg_score").isNotNull(),
            (1.0 - col("avg_score")) * 0.2
        ).otherwise(0.0)

        +

        when(
            col("avg_confidence_score").isNotNull(),
            (1.0 - col("avg_confidence_score")) * 0.1
        ).otherwise(0.0)
    )

    .withColumn(
        "weakness_sort_score",
        when(
            col("weakness_weight_sum") > 0,
            col("weakness_weighted_sum")
            / col("weakness_weight_sum")
        )
    )
)

In [19]:
weakness_rank_window = (
    Window
    .partitionBy("user_key", "date")
    .orderBy(
        col("weakness_sort_score").desc_nulls_last(),
        col("topic_key")
    )
)

agg_concept_weakness_daily_df = (
    concept_weakness_scored_df

    .withColumn(
        "weakness_rank",
        row_number()
        .over(weakness_rank_window)
        .cast("int")
    )

    .select(
        "user_key",
        "topic_key",
        "date",
        "failure_rate",
        "avg_score",
        "avg_hints_used",
        "repeated_mistake_count",
        "avg_confidence_score",
        "difficulty_score",
        "weakness_rank"
    )
)

agg_concept_weakness_daily_df.orderBy(
    "user_key",
    "date",
    "weakness_rank"
).show(truncate=False)

print(
    "Concept weakness rows:",
    agg_concept_weakness_daily_df.count()
)

+--------+---------+----------+------------+---------+--------------+----------------------+--------------------+----------------+-------------+
|user_key|topic_key|date      |failure_rate|avg_score|avg_hints_used|repeated_mistake_count|avg_confidence_score|difficulty_score|weakness_rank|
+--------+---------+----------+------------+---------+--------------+----------------------+--------------------+----------------+-------------+
|1       |5        |2026-07-20|0.5         |0.5      |0.5           |0                     |NULL                |NULL            |1            |
|1       |5        |2026-07-23|NULL        |NULL     |NULL          |0                     |0.4                 |NULL            |1            |
|1       |9        |2026-07-23|NULL        |NULL     |NULL          |0                     |0.7                 |0.0             |2            |
|1       |5        |2026-07-28|NULL        |NULL     |NULL          |0                     |0.4333              |0.675           |

In [20]:
from pyspark.sql.functions import col, when, row_number
from pyspark.sql.window import Window

weakness_rank_window = (
    Window
    .partitionBy("user_key", "date")
    .orderBy(
        col("weakness_sort_score").desc_nulls_last(),
        col("topic_key")
    )
)

ranked_concept_weakness_df = (
    concept_weakness_scored_df
    .withColumn(
        "temporary_weakness_rank",
        row_number().over(weakness_rank_window)
    )
)

agg_concept_weakness_daily_df = (
    ranked_concept_weakness_df
    .withColumn(
        "weakness_rank",
        when(
            col("weakness_sort_score").isNotNull(),
            col("temporary_weakness_rank")
        )
        .otherwise(None)
        .cast("int")
    )
    .select(
        "user_key",
        "topic_key",
        "date",
        "failure_rate",
        "avg_score",
        "avg_hints_used",
        "repeated_mistake_count",
        "avg_confidence_score",
        "difficulty_score",
        "weakness_rank"
    )
)

agg_concept_weakness_daily_df.orderBy(
    "user_key",
    "date",
    col("weakness_rank").asc_nulls_last(),
    "topic_key"
).show(truncate=False)

+--------+---------+----------+------------+---------+--------------+----------------------+--------------------+----------------+-------------+
|user_key|topic_key|date      |failure_rate|avg_score|avg_hints_used|repeated_mistake_count|avg_confidence_score|difficulty_score|weakness_rank|
+--------+---------+----------+------------+---------+--------------+----------------------+--------------------+----------------+-------------+
|1       |5        |2026-07-20|0.5         |0.5      |0.5           |0                     |NULL                |NULL            |1            |
|1       |5        |2026-07-23|NULL        |NULL     |NULL          |0                     |0.4                 |NULL            |1            |
|1       |9        |2026-07-23|NULL        |NULL     |NULL          |0                     |0.7                 |0.0             |2            |
|1       |5        |2026-07-28|NULL        |NULL     |NULL          |0                     |0.4333              |0.675           |

In [21]:
spark.sql("""
DELETE FROM demo.gold.agg_concept_weakness_daily
""")

DataFrame[]

In [22]:
agg_concept_weakness_daily_df.writeTo(
    "demo.gold.agg_concept_weakness_daily"
).append()

In [23]:
spark.sql("""
SELECT *
FROM demo.gold.agg_concept_weakness_daily
ORDER BY
    user_key,
    date,
    weakness_rank NULLS LAST,
    topic_key
""").show(truncate=False)

+--------+---------+----------+------------+---------+--------------+----------------------+--------------------+----------------+-------------+
|user_key|topic_key|date      |failure_rate|avg_score|avg_hints_used|repeated_mistake_count|avg_confidence_score|difficulty_score|weakness_rank|
+--------+---------+----------+------------+---------+--------------+----------------------+--------------------+----------------+-------------+
|1       |5        |2026-07-20|0.5         |0.5      |0.5           |0                     |NULL                |NULL            |1            |
|1       |5        |2026-07-23|NULL        |NULL     |NULL          |0                     |0.4                 |NULL            |1            |
|1       |9        |2026-07-23|NULL        |NULL     |NULL          |0                     |0.7                 |0.0             |2            |
|1       |5        |2026-07-28|NULL        |NULL     |NULL          |0                     |0.4333              |0.675           |

In [24]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(
        DISTINCT CONCAT(
            CAST(user_key AS STRING),
            '||',
            CAST(topic_key AS STRING),
            '||',
            CAST(date AS STRING)
        )
    ) AS distinct_learner_topic_dates
FROM demo.gold.agg_concept_weakness_daily
""").show()

+----------+----------------------------+
|total_rows|distinct_learner_topic_dates|
+----------+----------------------------+
|        17|                          17|
+----------+----------------------------+



In [25]:
from pyspark.sql.functions import (
    col,
    to_date,
    avg,
    count,
    sum,
    when
)

daily_practice_progress_df = (
    fact_attempt_df
    .groupBy(
        "user_key",
        to_date("attempt_time").alias("date")
    )
    .agg(
        avg("score")
        .cast("float")
        .alias("avg_practice_score"),

        count("*")
        .cast("int")
        .alias("total_attempts"),

        sum(
            when(
                col("is_correct") == True,
                1
            ).otherwise(0)
        )
        .cast("int")
        .alias("successful_attempts"),

        avg(
            when(
                col("hints_used") > 0,
                1.0
            ).otherwise(0.0)
        )
        .cast("float")
        .alias("hint_usage_rate")
    )
)

daily_practice_progress_df.orderBy(
    "user_key",
    "date"
).show(truncate=False)

+--------+----------+------------------+--------------+-------------------+---------------+
|user_key|date      |avg_practice_score|total_attempts|successful_attempts|hint_usage_rate|
+--------+----------+------------------+--------------+-------------------+---------------+
|1       |2026-07-20|0.5               |2             |1                  |0.5            |
|2       |2026-07-21|0.5               |2             |1                  |0.0            |
|3       |2026-07-22|1.0               |1             |1                  |0.0            |
+--------+----------+------------------+--------------+-------------------+---------------+



In [26]:
from pyspark.sql.functions import (
    col,
    to_date,
    avg
)

daily_mastery_progress_df = (
    fact_concept_state_df
    .filter(
        col("is_current") == True
    )
    .groupBy(
        "user_key",
        to_date("valid_from").alias("date")
    )
    .agg(
        avg("mastery_score")
        .cast("float")
        .alias("avg_mastery_score")
    )
)

daily_mastery_progress_df.orderBy(
    "user_key",
    "date"
).show(truncate=False)

+--------+----------+-----------------+
|user_key|date      |avg_mastery_score|
+--------+----------+-----------------+
|1       |2026-07-23|0.7              |
|1       |2026-07-28|0.4567           |
|2       |2026-07-28|0.5667           |
|3       |2026-07-22|0.9              |
|3       |2026-07-25|0.6              |
|3       |2026-07-28|0.9              |
+--------+----------+-----------------+



In [27]:
from pyspark.sql.functions import (
    col,
    to_date,
    avg
)

daily_confusion_progress_df = (
    fact_feedback_df
    .filter(
        col("still_confused").isNotNull()
    )
    .groupBy(
        "user_key",
        to_date("feedback_time").alias("date")
    )
    .agg(
        avg(
            col("still_confused").cast("int")
        )
        .cast("float")
        .alias("still_confused_rate")
    )
)

daily_confusion_progress_df.orderBy(
    "user_key",
    "date"
).show(truncate=False)

+--------+----------+-------------------+
|user_key|date      |still_confused_rate|
+--------+----------+-------------------+
|1       |2026-07-20|1.0                |
|1       |2026-07-23|0.5                |
|2       |2026-07-21|1.0                |
|2       |2026-07-24|1.0                |
|3       |2026-07-22|0.0                |
|3       |2026-07-25|0.0                |
+--------+----------+-------------------+



In [28]:
progress_daily_spine_df = (
    daily_practice_progress_df
    .select("user_key", "date")

    .unionByName(
        daily_mastery_progress_df
        .select("user_key", "date")
    )

    .unionByName(
        daily_confusion_progress_df
        .select("user_key", "date")
    )

    .filter(
        col("user_key").isNotNull()
        & col("date").isNotNull()
    )

    .distinct()
)

progress_daily_spine_df.orderBy(
    "user_key",
    "date"
).show(truncate=False)

print(
    "Progress learner-date rows:",
    progress_daily_spine_df.count()
)

+--------+----------+
|user_key|date      |
+--------+----------+
|1       |2026-07-20|
|1       |2026-07-23|
|1       |2026-07-28|
|2       |2026-07-21|
|2       |2026-07-24|
|2       |2026-07-28|
|3       |2026-07-22|
|3       |2026-07-25|
|3       |2026-07-28|
+--------+----------+

Progress learner-date rows: 9


In [29]:
from pyspark.sql.functions import coalesce, lit

agg_learning_progress_daily_df = (
    progress_daily_spine_df.alias("spine")

    .join(
        daily_practice_progress_df.alias("practice"),
        ["user_key", "date"],
        "left"
    )

    .join(
        daily_mastery_progress_df.alias("mastery"),
        ["user_key", "date"],
        "left"
    )

    .join(
        daily_confusion_progress_df.alias("confusion"),
        ["user_key", "date"],
        "left"
    )

    .select(
        col("user_key").cast("int"),
        col("date"),

        col("mastery.avg_mastery_score")
        .cast("float")
        .alias("avg_mastery_score"),

        col("practice.avg_practice_score")
        .cast("float")
        .alias("avg_practice_score"),

        coalesce(
            col("practice.total_attempts"),
            lit(0)
        )
        .cast("int")
        .alias("total_attempts"),

        coalesce(
            col("practice.successful_attempts"),
            lit(0)
        )
        .cast("int")
        .alias("successful_attempts"),

        col("practice.hint_usage_rate")
        .cast("float")
        .alias("hint_usage_rate"),

        col("confusion.still_confused_rate")
        .cast("float")
        .alias("still_confused_rate")
    )
)

agg_learning_progress_daily_df.orderBy(
    "user_key",
    "date"
).show(truncate=False)

+--------+----------+-----------------+------------------+--------------+-------------------+---------------+-------------------+
|user_key|date      |avg_mastery_score|avg_practice_score|total_attempts|successful_attempts|hint_usage_rate|still_confused_rate|
+--------+----------+-----------------+------------------+--------------+-------------------+---------------+-------------------+
|1       |2026-07-20|NULL             |0.5               |2             |1                  |0.5            |1.0                |
|1       |2026-07-23|0.7              |NULL              |0             |0                  |NULL           |0.5                |
|1       |2026-07-28|0.4567           |NULL              |0             |0                  |NULL           |NULL               |
|2       |2026-07-21|NULL             |0.5               |2             |1                  |0.0            |1.0                |
|2       |2026-07-24|NULL             |NULL              |0             |0                

In [30]:
spark.sql("""
DELETE FROM demo.gold.agg_learning_progress_daily
""")

DataFrame[]

In [31]:
agg_learning_progress_daily_df.writeTo(
    "demo.gold.agg_learning_progress_daily"
).append()

In [32]:
spark.sql("""
SELECT *
FROM demo.gold.agg_learning_progress_daily
ORDER BY user_key, date
""").show(truncate=False)

+--------+----------+-----------------+------------------+--------------+-------------------+---------------+-------------------+
|user_key|date      |avg_mastery_score|avg_practice_score|total_attempts|successful_attempts|hint_usage_rate|still_confused_rate|
+--------+----------+-----------------+------------------+--------------+-------------------+---------------+-------------------+
|1       |2026-07-20|NULL             |0.5               |2             |1                  |0.5            |1.0                |
|1       |2026-07-23|0.7              |NULL              |0             |0                  |NULL           |0.5                |
|1       |2026-07-28|0.4567           |NULL              |0             |0                  |NULL           |NULL               |
|2       |2026-07-21|NULL             |0.5               |2             |1                  |0.0            |1.0                |
|2       |2026-07-24|NULL             |NULL              |0             |0                

In [33]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(
        DISTINCT CONCAT(
            CAST(user_key AS STRING),
            '||',
            CAST(date AS STRING)
        )
    ) AS distinct_learner_dates
FROM demo.gold.agg_learning_progress_daily
""").show()

+----------+----------------------+
|total_rows|distinct_learner_dates|
+----------+----------------------+
|         9|                     9|
+----------+----------------------+



In [34]:
from pyspark.sql.functions import (
    col,
    avg,
    first
)

pre_practice_illusion_df = (
    fact_feedback_df
    .filter(
        col("feedback_stage") == "before_practice"
    )
    .select(
        "user_key",
        "session_id",
        "practice_id",

        col("confidence_score")
        .cast("int")
        .alias("confidence_before")
    )
)

post_practice_illusion_df = (
    fact_feedback_df
    .filter(
        col("feedback_stage") == "after_practice"
    )
    .select(
        "user_key",
        "session_id",
        "practice_id",

        col("confidence_score")
        .cast("int")
        .alias("confidence_after")
    )
)

practice_illusion_df = (
    fact_attempt_df
    .groupBy(
        "user_key",
        "session_id",
        "practice_id",
        "topic_key"
    )
    .agg(
        avg("score")
        .cast("float")
        .alias("practice_score")
    )
)

In [35]:
print("Confidence before practice")
pre_practice_illusion_df.orderBy(
    "user_key"
).show(truncate=False)

print("Practice performance")
practice_illusion_df.orderBy(
    "user_key"
).show(truncate=False)

print("Confidence after practice")
post_practice_illusion_df.orderBy(
    "user_key"
).show(truncate=False)

Confidence before practice
+--------+-----------+------------+-----------------+
|user_key|session_id |practice_id |confidence_before|
+--------+-----------+------------+-----------------+
|1       |session_001|practice_001|4                |
|2       |session_002|practice_002|9                |
|3       |session_003|practice_003|7                |
+--------+-----------+------------+-----------------+

Practice performance
+--------+-----------+------------+---------+--------------+
|user_key|session_id |practice_id |topic_key|practice_score|
+--------+-----------+------------+---------+--------------+
|1       |session_001|practice_001|5        |0.5           |
|2       |session_002|practice_002|9        |0.5           |
|3       |session_003|practice_003|8        |1.0           |
+--------+-----------+------------+---------+--------------+

Confidence after practice
+--------+-----------+------------+----------------+
|user_key|session_id |practice_id |confidence_after|
+--------+---

In [36]:
from pyspark.sql.functions import (
    col,
    round as spark_round,
    when
)

agg_illusion_of_learning_df = (
    pre_practice_illusion_df.alias("pre")

    .join(
        practice_illusion_df.alias("practice"),
        ["user_key", "session_id", "practice_id"],
        "inner"
    )

    .join(
        post_practice_illusion_df.alias("post"),
        ["user_key", "session_id", "practice_id"],
        "inner"
    )

    .select(
        col("user_key").cast("int"),
        col("practice.topic_key").cast("int"),
        col("session_id"),

        col("pre.confidence_before").cast("int"),
        col("practice.practice_score").cast("float"),
        col("post.confidence_after").cast("int"),

        (
            col("pre.confidence_before")
            - col("post.confidence_after")
        )
        .cast("int")
        .alias("confidence_drop"),

        spark_round(
            col("pre.confidence_before") / 10.0
            - col("practice.practice_score"),
            4
        )
        .cast("float")
        .alias("illusion_gap_score"),

        when(
            (
                col("pre.confidence_before") / 10.0
                - col("practice.practice_score")
            ) >= 0.30,
            True
        )
        .otherwise(False)
        .cast("boolean")
        .alias("illusion_flag")
    )
)

agg_illusion_of_learning_df.orderBy(
    "user_key"
).show(truncate=False)

+--------+---------+-----------+-----------------+--------------+----------------+---------------+------------------+-------------+
|user_key|topic_key|session_id |confidence_before|practice_score|confidence_after|confidence_drop|illusion_gap_score|illusion_flag|
+--------+---------+-----------+-----------------+--------------+----------------+---------------+------------------+-------------+
|1       |5        |session_001|4                |0.5           |5               |-1             |-0.1              |false        |
|2       |9        |session_002|9                |0.5           |5               |4              |0.4               |true         |
|3       |8        |session_003|7                |1.0           |9               |-2             |-0.3              |false        |
+--------+---------+-----------+-----------------+--------------+----------------+---------------+------------------+-------------+



In [37]:
spark.sql("""
DELETE FROM demo.gold.agg_illusion_of_learning
""")

DataFrame[]

In [38]:
agg_illusion_of_learning_df.writeTo(
    "demo.gold.agg_illusion_of_learning"
).append()

In [39]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(
        DISTINCT CONCAT(
            CAST(user_key AS STRING),
            '||',
            CAST(topic_key AS STRING),
            '||',
            session_id
        )
    ) AS distinct_learner_topic_sessions
FROM demo.gold.agg_illusion_of_learning
""").show()

+----------+-------------------------------+
|total_rows|distinct_learner_topic_sessions|
+----------+-------------------------------+
|         3|                              3|
+----------+-------------------------------+



In [40]:
fact_attempt_df = spark.table(
    "demo.gold.fact_practice_attempt"
)

fact_feedback_df = spark.table(
    "demo.gold.fact_learning_feedback"
)

fact_validation_df = spark.table(
    "demo.gold.fact_ai_insight_validation"
)

fact_concept_state_df = spark.table(
    "demo.gold.fact_learner_concept_state"
)

illusion_df = spark.table(
    "demo.gold.agg_illusion_of_learning"
)

print("Practice attempts:", fact_attempt_df.count())
print("Feedback rows:", fact_feedback_df.count())
print("AI validations:", fact_validation_df.count())
print("Concept states:", fact_concept_state_df.count())
print("Illusion records:", illusion_df.count())

Practice attempts: 5
Feedback rows: 11
AI validations: 9
Concept states: 12
Illusion records: 3


In [41]:
from pyspark.sql.functions import (
    col,
    min as spark_min
)

prediction_sessions_df = (
    fact_attempt_df
    .groupBy(
        "user_key",
        "topic_key",
        "session_id"
    )
    .agg(
        spark_min("attempt_time")
        .alias("prediction_time")
    )
)

prediction_sessions_df.orderBy(
    "user_key",
    "topic_key",
    "session_id"
).show(truncate=False)

print(
    "Prediction sessions:",
    prediction_sessions_df.count()
)

+--------+---------+-----------+-------------------+
|user_key|topic_key|session_id |prediction_time    |
+--------+---------+-----------+-------------------+
|1       |5        |session_001|2026-07-20 09:12:00|
|2       |9        |session_002|2026-07-21 11:11:00|
|3       |8        |session_003|2026-07-22 14:08:00|
+--------+---------+-----------+-------------------+

Prediction sessions: 3


In [42]:
from pyspark.sql.functions import (
    col,
    expr,
    avg,
    sum,
    when,
    count
)

historical_practice_7d_df = (
    prediction_sessions_df.alias("p")

    .join(
        fact_attempt_df.alias("a"),
        expr("""
            a.user_key = p.user_key
            AND a.topic_key = p.topic_key
            AND a.attempt_time < p.prediction_time
            AND a.attempt_time >= p.prediction_time - INTERVAL 7 DAYS
        """),
        "left"
    )

    .groupBy(
        col("p.user_key"),
        col("p.topic_key"),
        col("p.session_id"),
        col("p.prediction_time")
    )

    .agg(
        count("a.attempt_id")
        .cast("int")
        .alias("historical_attempt_count"),

        avg("a.score")
        .cast("float")
        .alias("avg_score_last_7_days"),

        avg(
            when(
                col("a.is_correct") == False,
                1.0
            ).when(
                col("a.is_correct") == True,
                0.0
            )
        )
        .cast("float")
        .alias("failure_rate_last_7_days"),

        sum("a.hints_used")
        .cast("int")
        .alias("hints_used_last_7_days"),

        avg("a.attempt_duration_seconds")
        .cast("float")
        .alias("avg_attempt_duration")
    )
)

historical_practice_7d_df.orderBy(
    "user_key",
    "topic_key",
    "session_id"
).show(truncate=False)

+--------+---------+-----------+-------------------+------------------------+---------------------+------------------------+----------------------+--------------------+
|user_key|topic_key|session_id |prediction_time    |historical_attempt_count|avg_score_last_7_days|failure_rate_last_7_days|hints_used_last_7_days|avg_attempt_duration|
+--------+---------+-----------+-------------------+------------------------+---------------------+------------------------+----------------------+--------------------+
|1       |5        |session_001|2026-07-20 09:12:00|0                       |NULL                 |NULL                    |NULL                  |NULL                |
|2       |9        |session_002|2026-07-21 11:11:00|0                       |NULL                 |NULL                    |NULL                  |NULL                |
|3       |8        |session_003|2026-07-22 14:08:00|0                       |NULL                 |NULL                    |NULL                  |NULL    

In [43]:
from pyspark.sql.functions import (
    col,
    expr,
    avg
)

pre_prediction_feedback_df = (
    prediction_sessions_df.alias("p")

    .join(
        fact_feedback_df.alias("f"),
        expr("""
            f.user_key = p.user_key
            AND f.session_id = p.session_id
            AND f.feedback_stage = 'before_practice'
            AND f.feedback_time < p.prediction_time
        """),
        "left"
    )

    .groupBy(
        col("p.user_key"),
        col("p.topic_key"),
        col("p.session_id"),
        col("p.prediction_time")
    )

    .agg(
        avg("f.confidence_score")
        .cast("float")
        .alias("confidence_before_avg")
    )
)

pre_prediction_feedback_df.orderBy(
    "user_key",
    "topic_key",
    "session_id"
).show(truncate=False)

+--------+---------+-----------+-------------------+---------------------+
|user_key|topic_key|session_id |prediction_time    |confidence_before_avg|
+--------+---------+-----------+-------------------+---------------------+
|1       |5        |session_001|2026-07-20 09:12:00|4.0                  |
|2       |9        |session_002|2026-07-21 11:11:00|9.0                  |
|3       |8        |session_003|2026-07-22 14:08:00|7.0                  |
+--------+---------+-----------+-------------------+---------------------+



In [44]:
from pyspark.sql.functions import (
    col,
    expr,
    avg
)

historical_post_feedback_df = (
    prediction_sessions_df.alias("p")

    .join(
        fact_feedback_df.alias("f"),
        expr("""
            f.user_key = p.user_key
            AND f.feedback_stage = 'after_practice'
            AND f.feedback_time < p.prediction_time
            AND f.session_id <> p.session_id
        """),
        "left"
    )

    .groupBy(
        col("p.user_key"),
        col("p.topic_key"),
        col("p.session_id"),
        col("p.prediction_time")
    )

    .agg(
        avg("f.confidence_score")
        .cast("float")
        .alias("confidence_after_avg"),

        avg(
            col("f.still_confused").cast("int")
        )
        .cast("float")
        .alias("still_confused_rate")
    )
)

historical_post_feedback_df.orderBy(
    "user_key",
    "topic_key",
    "session_id"
).show(truncate=False)

+--------+---------+-----------+-------------------+--------------------+-------------------+
|user_key|topic_key|session_id |prediction_time    |confidence_after_avg|still_confused_rate|
+--------+---------+-----------+-------------------+--------------------+-------------------+
|1       |5        |session_001|2026-07-20 09:12:00|NULL                |NULL               |
|2       |9        |session_002|2026-07-21 11:11:00|NULL                |NULL               |
|3       |8        |session_003|2026-07-22 14:08:00|NULL                |NULL               |
+--------+---------+-----------+-------------------+--------------------+-------------------+



In [45]:
from pyspark.sql.functions import (
    col,
    expr,
    avg
)

historical_ai_validation_df = (
    prediction_sessions_df.alias("p")

    .join(
        fact_validation_df.alias("v"),
        expr("""
            v.user_key = p.user_key
            AND v.topic_key = p.topic_key
            AND v.validation_time < p.prediction_time
        """),
        "left"
    )

    .groupBy(
        col("p.user_key"),
        col("p.topic_key"),
        col("p.session_id"),
        col("p.prediction_time")
    )

    .agg(
        avg("v.extraction_confidence")
        .cast("float")
        .alias("extraction_confidence_avg"),

        avg("v.reliability_score")
        .cast("float")
        .alias("reliability_score_avg")
    )
)

historical_ai_validation_df.orderBy(
    "user_key",
    "topic_key",
    "session_id"
).show(truncate=False)

+--------+---------+-----------+-------------------+-------------------------+---------------------+
|user_key|topic_key|session_id |prediction_time    |extraction_confidence_avg|reliability_score_avg|
+--------+---------+-----------+-------------------+-------------------------+---------------------+
|1       |5        |session_001|2026-07-20 09:12:00|NULL                     |NULL                 |
|2       |9        |session_002|2026-07-21 11:11:00|NULL                     |NULL                 |
|3       |8        |session_003|2026-07-22 14:08:00|NULL                     |NULL                 |
+--------+---------+-----------+-------------------+-------------------------+---------------------+



In [46]:
from pyspark.sql.functions import col, expr, row_number
from pyspark.sql.window import Window

historical_concept_state_candidates_df = (
    prediction_sessions_df.alias("p")
    .join(
        fact_concept_state_df.alias("s"),
        expr("""
            s.user_key = p.user_key
            AND s.topic_key = p.topic_key
            AND s.valid_from < p.prediction_time
            AND (
                s.valid_to IS NULL
                OR s.valid_to >= p.prediction_time
            )
        """),
        "left"
    )
)

historical_state_window = (
    Window
    .partitionBy(
        col("p.user_key"),
        col("p.topic_key"),
        col("p.session_id")
    )
    .orderBy(
        col("s.valid_from").desc_nulls_last(),
        col("s.state_version").desc_nulls_last()
    )
)

historical_concept_state_df = (
    historical_concept_state_candidates_df
    .withColumn(
        "state_row_number",
        row_number().over(historical_state_window)
    )
    .filter(col("state_row_number") == 1)
    .select(
        col("p.user_key"),
        col("p.topic_key"),
        col("p.session_id"),
        col("p.prediction_time"),

        col("s.mastery_score")
            .cast("float")
            .alias("historical_mastery_score"),

        col("s.difficulty_score")
            .cast("float")
            .alias("historical_difficulty_score"),

        col("s.repeated_mistake_count")
            .cast("int")
            .alias("repeated_mistake_count")
    )
)

historical_concept_state_df.orderBy(
    "user_key",
    "topic_key",
    "session_id"
).show(truncate=False)

+--------+---------+-----------+-------------------+------------------------+---------------------------+----------------------+
|user_key|topic_key|session_id |prediction_time    |historical_mastery_score|historical_difficulty_score|repeated_mistake_count|
+--------+---------+-----------+-------------------+------------------------+---------------------------+----------------------+
|1       |5        |session_001|2026-07-20 09:12:00|NULL                    |NULL                       |NULL                  |
|2       |9        |session_002|2026-07-21 11:11:00|NULL                    |NULL                       |NULL                  |
|3       |8        |session_003|2026-07-22 14:08:00|NULL                    |NULL                       |NULL                  |
+--------+---------+-----------+-------------------+------------------------+---------------------------+----------------------+



In [50]:
from pyspark.sql.functions import when

In [52]:
from pyspark.sql.functions import (
    col,
    expr,
    avg,
    min as spark_min
)

# זמן ההתחלה של כל סשן תרגול
practice_session_times_df = (
    fact_attempt_df
    .groupBy(
        "user_key",
        "topic_key",
        "session_id"
    )
    .agg(
        spark_min("attempt_time")
        .alias("historical_session_time")
    )
)

# הוספת זמן הסשן לכל רשומת illusion
illusion_with_time_df = (
    illusion_df.alias("i")
    .join(
        practice_session_times_df.alias("t"),
        ["user_key", "topic_key", "session_id"],
        "inner"
    )
    .select(
        "user_key",
        "topic_key",
        "session_id",
        "illusion_gap_score",
        "historical_session_time"
    )
)

# שימוש רק בסשנים קודמים לאותה נקודת חיזוי
historical_illusion_df = (
    prediction_sessions_df.alias("p")

    .join(
        illusion_with_time_df.alias("i"),
        expr("""
            i.user_key = p.user_key
            AND i.topic_key = p.topic_key
            AND i.session_id <> p.session_id
            AND i.historical_session_time < p.prediction_time
        """),
        "left"
    )

    .groupBy(
        col("p.user_key"),
        col("p.topic_key"),
        col("p.session_id"),
        col("p.prediction_time")
    )

    .agg(
        avg("i.illusion_gap_score")
        .cast("float")
        .alias("illusion_gap_score")
    )
)

historical_illusion_df.orderBy(
    "user_key",
    "topic_key",
    "session_id"
).show(truncate=False)

+--------+---------+-----------+-------------------+------------------+
|user_key|topic_key|session_id |prediction_time    |illusion_gap_score|
+--------+---------+-----------+-------------------+------------------+
|1       |5        |session_001|2026-07-20 09:12:00|NULL              |
|2       |9        |session_002|2026-07-21 11:11:00|NULL              |
|3       |8        |session_003|2026-07-22 14:08:00|NULL              |
+--------+---------+-----------+-------------------+------------------+



In [53]:
from pyspark.sql.functions import (
    col,
    coalesce,
    lit
)

ml_learning_difficulty_features_df = (
    prediction_sessions_df.alias("p")

    .join(
        historical_practice_7d_df.alias("practice"),
        ["user_key", "topic_key", "session_id", "prediction_time"],
        "left"
    )

    .join(
        pre_prediction_feedback_df.alias("pre"),
        ["user_key", "topic_key", "session_id", "prediction_time"],
        "left"
    )

    .join(
        historical_post_feedback_df.alias("post"),
        ["user_key", "topic_key", "session_id", "prediction_time"],
        "left"
    )

    .join(
        historical_illusion_df.alias("illusion"),
        ["user_key", "topic_key", "session_id", "prediction_time"],
        "left"
    )

    .join(
        historical_concept_state_df.alias("state"),
        ["user_key", "topic_key", "session_id", "prediction_time"],
        "left"
    )

    .join(
        historical_ai_validation_df.alias("ai"),
        ["user_key", "topic_key", "session_id", "prediction_time"],
        "left"
    )

    .select(
        col("user_key").cast("int"),
        col("topic_key").cast("int"),
        col("session_id"),

        col("practice.avg_score_last_7_days")
            .cast("float"),

        col("practice.failure_rate_last_7_days")
            .cast("float"),

        coalesce(
            col("practice.hints_used_last_7_days"),
            lit(0)
        )
        .cast("int")
        .alias("hints_used_last_7_days"),

        col("practice.avg_attempt_duration")
            .cast("float"),

        col("pre.confidence_before_avg")
            .cast("float"),

        col("post.confidence_after_avg")
            .cast("float"),

        col("post.still_confused_rate")
            .cast("float"),

        col("illusion.illusion_gap_score")
            .cast("float"),

        coalesce(
            col("state.repeated_mistake_count"),
            lit(0)
        )
        .cast("int")
        .alias("repeated_mistake_count"),

        col("ai.extraction_confidence_avg")
            .cast("float"),

        col("ai.reliability_score_avg")
            .cast("float"),

        lit(None)
            .cast("float")
            .alias("overall_motivation_avg"),

        lit(None)
            .cast("float")
            .alias("overall_stress_avg"),

        lit(None)
            .cast("float")
            .alias("topic_self_reported_understanding_avg"),

        col("pre.confidence_before_avg")
            .cast("float")
            .alias("topic_confidence_avg")
    )
)

ml_learning_difficulty_features_df.orderBy(
    "user_key",
    "topic_key",
    "session_id"
).show(truncate=False)

+--------+---------+-----------+---------------------+------------------------+----------------------+--------------------+---------------------+--------------------+-------------------+------------------+----------------------+-------------------------+---------------------+----------------------+------------------+-------------------------------------+--------------------+
|user_key|topic_key|session_id |avg_score_last_7_days|failure_rate_last_7_days|hints_used_last_7_days|avg_attempt_duration|confidence_before_avg|confidence_after_avg|still_confused_rate|illusion_gap_score|repeated_mistake_count|extraction_confidence_avg|reliability_score_avg|overall_motivation_avg|overall_stress_avg|topic_self_reported_understanding_avg|topic_confidence_avg|
+--------+---------+-----------+---------------------+------------------------+----------------------+--------------------+---------------------+--------------------+-------------------+------------------+----------------------+----------------

In [54]:
spark.sql("""
DELETE FROM demo.gold.ml_learning_difficulty_features
""")

DataFrame[]

In [55]:
ml_learning_difficulty_features_df.writeTo(
    "demo.gold.ml_learning_difficulty_features"
).append()

In [56]:
spark.sql("""
SELECT *
FROM demo.gold.ml_learning_difficulty_features
ORDER BY user_key, topic_key, session_id
""").show(truncate=False)

+--------+---------+-----------+---------------------+------------------------+----------------------+--------------------+---------------------+--------------------+-------------------+------------------+----------------------+-------------------------+---------------------+----------------------+------------------+-------------------------------------+--------------------+
|user_key|topic_key|session_id |avg_score_last_7_days|failure_rate_last_7_days|hints_used_last_7_days|avg_attempt_duration|confidence_before_avg|confidence_after_avg|still_confused_rate|illusion_gap_score|repeated_mistake_count|extraction_confidence_avg|reliability_score_avg|overall_motivation_avg|overall_stress_avg|topic_self_reported_understanding_avg|topic_confidence_avg|
+--------+---------+-----------+---------------------+------------------------+----------------------+--------------------+---------------------+--------------------+-------------------+------------------+----------------------+----------------